# 03 -- Region Analysis

**Purpose:** Compare Konkan, Western Maharashtra, Marathwada, and Vidarbha
on Shala Score v2, GPI, marginalised enrolment, and pillar scores.

**Prerequisites:** `src/core/regions.py` and `build_shala_score_v2.py`.

**Region counts:** Konkan=7, Western Maharashtra=10, Marathwada=8, Vidarbha=11 (total 36).

**Public API from `src/core/regions.py`:**
- `REGION_MAP` -- dict of district -> region
- `REGIONS` -- list of four region names
- `add_region(df)` -- adds 'region' column; raises ValueError on unknown districts
- `compute_gpi(df)` -- adds 'gpi' column to DataFrame
- `compute_marginalised_enrolment_pct(df)` -- adds 'marginalised_enrolment_pct' column

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

# Correct imports -- REGION_MAP and REGIONS, not REGION_DISTRICTS
from src.core.regions import add_region, REGION_MAP, REGIONS, compute_gpi

V2 = ROOT / 'data/processed/district_year/maharashtra_district_year_shala_score_v2.csv'
df = pd.read_csv(V2, low_memory=False)
df = add_region(df)  # raises ValueError if any district is unmapped

print('Region distribution (district-year rows):')
print(df['region'].value_counts().to_string())
print(f'\nAll 4 regions present: {set(df["region"].unique()) == set(REGIONS)}')

In [ ]:
# Unique district count per region (must match Konkan=7, W.Mah=10, Marathwada=8, Vidarbha=11)
dist_per_region = df.groupby('region')['district'].nunique().sort_values(ascending=False)
print('Districts per region (expected: Vidarbha=11, W.Mah=10, Marathwada=8, Konkan=7):')
print(dist_per_region.to_string())

In [ ]:
# Region mean Shala Score v2 by year
pivot = df.groupby(['academic_year', 'region'])['shala_score_v2'].mean().unstack()
pivot = pivot.reindex(sorted(pivot.index,
                             key=lambda x: int(str(x)[:4])))
pivot.plot(marker='o', figsize=(11, 5))
plt.title('Mean Shala Score v2 by Region and Year')
plt.ylabel('Shala Score v2')
plt.xlabel('Academic Year')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Risk tier distribution by region (latest year)
latest = df['academic_year'].max()
print(f'Risk tier distribution by region in {latest}:')
print(
    df[df['academic_year'] == latest]
    .groupby(['region', 'risk_tier_v2'])['district']
    .count()
    .unstack(fill_value=0)
    .to_string()
)

In [ ]:
# Pillar means by region (all years)
pillar_cols = ['pillar2_teacher_ptr', 'pillar2_infrastructure',
               'pillar2_participation', 'pillar2_digital']
region_pillars = df.groupby('region')[pillar_cols].mean().round(1)
print('Mean pillar scores by region (all years):')
print(region_pillars.to_string())

In [ ]:
# GPI by region (compute from real enrolment columns)
df_gpi = compute_gpi(df)  # adds 'gpi' column
print('Mean GPI by region:')
print(df_gpi.groupby('region')['gpi'].mean().round(3).to_string())
print(f'\nGPI range overall: {df_gpi["gpi"].min():.3f} - {df_gpi["gpi"].max():.3f}')
print('(Expected: 0.80 - 0.98)')